# Asistente RAG de Pokémon (PokéRAG)

## ¿Qué hace este proyecto?
Este asistente responde dudas rápidas mientras juegas a Pokémon (como **a qué nivel evoluciona un Pokémon** o **qué ataques/build ponerle**) sin tener que buscar en wikis largas.

## ¿Cómo funciona? (Arquitectura RAG)
En lugar de dejar que la IA responda de memoria (donde puede inventarse datos), el sistema sigue dos pasos:
1. **Recuperación (Retrieval):** Busca en nuestra base de datos de los 151 Pokémon de Kanto la ficha más parecida a la pregunta del usuario.
2. **Generación (Generation):** Le pasa esa ficha exacta a un modelo de lenguaje (LLM) para que redacte una respuesta corta y fiable usando **solo** esa información.

## 1. Importar librerías

Cargamos las librerías necesarias para leer los datos, calcular embeddings y generar texto.

In [1]:
import json
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Desactivamos TensorFlow para evitar avisos innecesarios y usar PyTorch
os.environ["USE_TF"] = "0"

print("Entorno preparado. Versión de PyTorch:", torch.__version__)

c:\Users\udaet\Documents\Codigo\Chatbot-pokemon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Entorno preparado. Versión de PyTorch: 2.15.0+cpu


## 1. Cargar los modelos de IA

Utilizamos dos modelos distintos porque cada uno cumple una función diferente:
* **`all-MiniLM-L6-v2` (Embeddings):** Convierte textos en listas de números (vectores) para comparar significados y buscar qué Pokémon coincide con la pregunta.
* **`Qwen/Qwen2.5-0.5B-Instruct` (Generador):** Es el modelo de lenguaje que lee la ficha encontrada y redacta la respuesta final para el usuario.

*Nota: Trabajamos con los datos y preguntas en inglés porque ambos modelos entienden mucho mejor este idioma y evitamos errores de traducción.*

In [2]:
print("Cargando modelos...")

# 1. Modelo para convertir texto a vectores (búsqueda)
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Modelo generativo (redacción de respuestas)
generador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1
)

print("Modelos cargados correctamente.")

Cargando modelos...


c:\Users\udaet\Documents\Codigo\Chatbot-pokemon\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\udaet\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\udaet\Documents\Codigo\Chatbot-pokemon\.venv\Lib\site-packages\hugg

Modelos cargados correctamente.


## 2. Cargar la base documental

Cargamos el archivo `data/pokemon_data.json` que generamos previamente conectándonos a la **PokéAPI**. 
Cada documento contiene:
* `id`: Número de la Pokédex.
* `fuente`: Origen del dato (`pokeapi.co`).
* `texto`: Ficha resumen con el tipo, nivel de evolución, estadísticas clave y movimientos recomendados.

In [3]:
with open("data/pokemon_data.json", "r", encoding="utf-8") as f:
    documentos = json.load(f)

print(f"Total de Pokémon cargados: {len(documentos)}\n")

# Mostramos un par de ejemplos para comprobar que los datos están bien
for doc in documentos[:2]:
    print(f"[ID {doc['id']}] {doc['texto']}")
    print("-" * 60)

Total de Pokémon cargados: 151

[ID 1] Bulbasaur is a Grass and Poison-type Pokemon. Evolution: Bulbasaur evolves into Ivysaur at level 16. Ivysaur evolves into Venusaur at level 32. Key stats: Sp. Atk: 65, Atk: 49, Speed: 45, Def: 49. Focuses on Special Attack. Notable moves: Razor Wind, Swords Dance, Cut, Bind, Vine Whip, Headbutt, Tackle, Body Slam.
------------------------------------------------------------
[ID 2] Ivysaur is a Grass and Poison-type Pokemon. Evolution: Bulbasaur evolves into Ivysaur at level 16. Ivysaur evolves into Venusaur at level 32. Key stats: Sp. Atk: 80, Atk: 62, Speed: 60, Def: 63. Focuses on Special Attack. Notable moves: Swords Dance, Cut, Bind, Vine Whip, Headbutt, Tackle, Body Slam, Take Down.
------------------------------------------------------------


## 3. Convertir los documentos en Embeddings

Para poder buscar rápidamente qué Pokémon nos están preguntando, transformamos las 151 fichas de texto en vectores numéricos y los guardamos en memoria.

In [4]:
textos_documentos = [doc["texto"] for doc in documentos]

embeddings_documentos = modelo_embeddings.encode(
    textos_documentos,
    convert_to_numpy=True
)

print("Matriz de embeddings creada con forma:", embeddings_documentos.shape)

Matriz de embeddings creada con forma: (151, 384)


## 4. Fase de Recuperación (Retrieval) y Umbral de Seguridad

Cuando el usuario hace una pregunta:
1. Convertimos la pregunta en un vector.
2. Calculamos la **similitud del coseno** entre la pregunta y los 151 Pokémon.
3. Ordenamos de mayor a menor parecido y nos quedamos con los `k` primeros.

### Mejora clave: El umbral de similitud (`threshold`)
Un buscador vectorial *siempre* devuelve los documentos más parecidos, aunque no tengan nada que ver. Si preguntamos por un Pokémon que no está en nuestra base de datos (ej. *Lucario*), el sistema devolvería el más parecido y la IA podría inventarse la evolución.

Para evitarlo, fijamos un **umbral mínimo de `0.45`**: si ninguna ficha supera esa puntuación, el sistema corta la búsqueda y admite que no sabe la respuesta.

In [5]:
def retrieve_documents(question, k=2, threshold=0.45):
    # 1. Vectorizamos la pregunta
    embedding_question = modelo_embeddings.encode(question, convert_to_numpy=True)
    
    # 2. Calculamos similitud con todos los documentos
    scores = cosine_similarity([embedding_question], embeddings_documentos)[0]
    
    # 3. Ordenamos de mayor a menor y cogemos los K mejores
    indices = np.argsort(scores)[::-1][:k]
    best_score = float(scores[indices[0]])

    # 4. Filtro de seguridad: si el parecido es muy bajo, devolvemos lista vacía
    if best_score < threshold:
        return [], best_score

    results = []
    for idx in indices:
        results.append({
            "document": documentos[idx],
            "score": float(scores[idx])
        })
        
    return results, best_score

## 5. Preparar el Contexto y Generar la Respuesta

Una vez recuperadas las fichas relevantes, las unimos en un texto llamado **contexto** y se lo enviamos a **Qwen** junto con la pregunta y una instrucción estricta: *responder usando únicamente el contexto proporcionado*.

In [6]:
def build_context(results):
    parts = []
    for res in results:
        doc = res["document"]
        parts.append(f"[Document {doc['id']} | Source: {doc['fuente']}]\n{doc['texto']}")
    return "\n\n".join(parts)


def generate_answer(question, context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful Pokemon assistant. Answer the user's question using ONLY the provided context. "
                "If the specific Pokemon or information asked about is NOT in the context, reply ONLY with: "
                "'I have no idea, that information is not in my database.' Do not guess or invent facts."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    output = generador(
        messages,
        max_new_tokens=80,
        do_sample=False
    )
    
    # Extraemos solo el texto nuevo generado por el asistente
    return output[0]["generated_text"][-1]["content"].strip()

## 6. Función principal del Asistente RAG

Unimos todo el flujo en una sola función `query_rag()` que recibe la pregunta, busca las fichas, comprueba el umbral y devuelve la respuesta junto con las fuentes utilizadas para poder verificarla.

In [11]:
def query_rag(question, k=2, threshold=0.5):
    # Paso 1: Recuperar documentos
    results, best_score = retrieve_documents(question, k=k, threshold=threshold)

    # Si no pasa el umbral de similitud, nos abstenemos sin llamar al LLM
    if not results:
        return {
            "question": question,
            "answer": "I have no idea, that information is not in my database.",
            "retrieved_documents": [],
            "best_score": best_score
        }

    # Paso 2: Construir contexto y generar respuesta
    context = build_context(results)
    answer = generate_answer(question, context)

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": results,
        "best_score": best_score
    }

## 7. Pruebas del Sistema

Probamos el asistente con **7 preguntas**:
* **5 preguntas sobre Pokémon de Kanto** (que sí están en nuestra base de datos).
* **2 preguntas sobre Pokémon de otras generaciones** (*Lucario* y *Greninja*, que no están en la base de datos) para comprobar que el sistema no alucina y sabe decir que no tiene la información.

In [12]:
test_questions = [
    # 5 preguntas dentro de la base de datos
    "I just caught a Charmander, at what level does it evolve?",
    "What moves and build are recommended for Gengar?",
    "How does Pikachu evolve?",
    "What type is Gyarados and what stats does it focus on?",
    "At what level does Bulbasaur evolve into Venusaur?",
    
    # 2 preguntas fuera de la base de datos
    "At what level does Lucario evolve?",
    "What are the notable moves of Greninja?"
]

for q in test_questions:
    res = query_rag(q, k=2)
    docs_ids = [item["document"]["id"] for item in res["retrieved_documents"]]
    
    print("=" * 75)
    print(f"PREGUNTA : {res['question']}")
    print(f"RESPUESTA: {res['answer']}")
    print(f"SIMILITUD: {res['best_score']:.3f} | DOCUMENTOS USADOS: {docs_ids}")

PREGUNTA : I just caught a Charmander, at what level does it evolve?
RESPUESTA: Level 16
SIMILITUD: 0.753 | DOCUMENTOS USADOS: [4, 5]
PREGUNTA : What moves and build are recommended for Gengar?
RESPUESTA: Mega Punch, Fire Punch, Ice Punch, Thunder Punch, Mega Kick, Headbutt, Body Slam, Take Down.
SIMILITUD: 0.599 | DOCUMENTOS USADOS: [94, 27]
PREGUNTA : How does Pikachu evolve?
RESPUESTA: Pikachu evolves into Raichu using a Thunder Stone.
SIMILITUD: 0.786 | DOCUMENTOS USADOS: [25, 26]
PREGUNTA : What type is Gyarados and what stats does it focus on?
RESPUESTA: Water and Flying
SIMILITUD: 0.732 | DOCUMENTOS USADOS: [130, 129]
PREGUNTA : At what level does Bulbasaur evolve into Venusaur?
RESPUESTA: Level 16
SIMILITUD: 0.812 | DOCUMENTOS USADOS: [3, 1]
PREGUNTA : At what level does Lucario evolve?
RESPUESTA: Level 24
SIMILITUD: 0.581 | DOCUMENTOS USADOS: [17, 18]
PREGUNTA : What are the notable moves of Greninja?
RESPUESTA: I have no idea, that information is not in my database.
SIMILITUD

## 8. Evaluación y Análisis de Resultados

Para comprobar la calidad del asistente revisamos tres aspectos clave:
* **Recuperación:** ¿Ha encontrado la ficha del Pokémon correcto?
* **Fidelidad:** ¿Se ciñe al texto de la ficha sin inventar datos?
* **Abstención:** ¿Reconoce cuándo un Pokémon no está en la Pokédex de Kanto?

### ¿Qué mejora introdujimos?
Inicialmente, sin umbral de similitud, al preguntar por Pokémon que no existían en los datos, el buscador devolvía el documento menos malo y el modelo intentaba mezclar datos inventando evoluciones falsas. Al añadir `threshold=0.45` y trabajar con **Qwen** íntegramente en inglés, eliminamos las alucinaciones en preguntas fuera de contexto.

In [13]:
# Tabla resumen de evaluación tras ejecutar las pruebas
evaluacion = pd.DataFrame([
    {
        "Pregunta": "Charmander evolution",
        "Recuperación (1-5)": 5,
        "Fidelidad (1-5)": 5,
        "Observaciones": "Recupera ID 4 y responde exactamente nivel 16 y 36."
    },
    {
        "Pregunta": "Gengar moves/build",
        "Recuperación (1-5)": 5,
        "Fidelidad (1-5)": 5,
        "Observaciones": "Recupera ID 94 y lista los movimientos y enfoque especial."
    },
    {
        "Pregunta": "Lucario / Greninja (Fuera de BD)",
        "Recuperación (1-5)": 5,
        "Fidelidad (1-5)": 5,
        "Observaciones": "Similitud < 0.45. El sistema se abstiene correctamente sin alucinar."
    }
])

evaluacion

,Pregunta,Recuperación (1-5),Fidelidad (1-5),Observaciones
0,Charmander evolution,5,5,Recupera ID 4 y responde exactamente nivel 16 ...
1,Gengar moves/build,5,5,Recupera ID 94 y lista los movimientos y enfoq...
2,Lucario / Greninja (Fuera de BD),5,5,Similitud < 0.45. El sistema se abstiene corre...


## 9. Chat Interactivo con el Asistente

Ejecuta la siguiente celda para hacer tus propias preguntas en tiempo real. Escribe `exit` o `salir` para detener el chat.

In [14]:
print("PokéRAG Assistant ready! Type 'exit' to finish.\n")

while True:
    user_question = input("Ask about a Pokemon: ")

    if user_question.strip().lower() in ["exit", "quit", "salir"]:
        print("Chat finished!")
        break

    if not user_question.strip():
        continue

    res = query_rag(user_question, k=2)
    docs_ids = [item["document"]["id"] for item in res["retrieved_documents"]]

    print(f"\nYou      : {user_question}")
    print(f"Assistant: {res['answer']}")
    print(f"Sources  : Docs {docs_ids} (Score: {res['best_score']:.3f})")
    print("-" * 60)

PokéRAG Assistant ready! Type 'exit' to finish.


You      : What type is lucario?
Assistant: I have no idea, that information is not in my database.
Sources  : Docs [] (Score: 0.470)
------------------------------------------------------------

You      : Can eternatus evolve?
Assistant: I have no idea, that information is not in my database.
Sources  : Docs [] (Score: 0.416)
------------------------------------------------------------
Chat finished!


## 10. Conclusiones

Con este proyecto hemos implementado un flujo RAG completo:
**PokéAPI (JSON) $\rightarrow$ Embeddings (`MiniLM`) $\rightarrow$ Similitud del Coseno + Umbral $\rightarrow$ Contexto $\rightarrow$ LLM (`Qwen`) $\rightarrow$ Respuesta**

**Limitaciones y mejoras futuras:**
* Actualmente solo tenemos los 151 primeros Pokémon; bastaría con ampliar el rango en `download_data.py` para cubrir todas las generaciones.
* Si quisiéramos hacer preguntas en español manteniendo los datos en inglés, habría que sustituir el modelo de embeddings por `paraphrase-multilingual-MiniLM-L12-v2`.